In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import gensim.downloader as api
from datasets import load_dataset
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.utils import compute_class_weight
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from task6.torch_utils.model_callbacks import EarlyStopping, ModelCheckpoint
from task6.utils.balance_dataset import augment_minority_classes
from task6.utils.prepare_data import prepare_data
from task6.utils.prepare_rnn_data import prepare_rnn_data

C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Szymon\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
import torch
import torchvision

print("Torch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


Torch version: 2.8.0+cu129
Torchvision version: 0.23.0+cu129
CUDA available: True
CUDA device: NVIDIA GeForce RTX 5080


In [3]:
df = pd.read_csv("../../data/merged_transcripts.csv")
df.head()

,Start Time,End Time,Sentence,Translation,Emotion_fine,Emotion_core,Intensity
0,1900-01-01 00:00:00,1900-01-01 00:00:04,"Ако приемаш моите извинения, просто ме пригърни.","If you accept my apologies, just hug me.",remorse,sadness,moderate
1,1900-01-01 00:00:06,1900-01-01 00:00:08,"Трябвам това, че си ме пригърни.",I need you to hug me.,longing,sadness,moderate
2,1900-01-01 00:00:08,1900-01-01 00:00:11,С теб всеки миски е прекрасен. Обичам.,"With you, every moment is wonderful. I love.",affection,happiness,moderate
3,1900-01-01 00:00:11,1900-01-01 00:00:16,Аз наистина ги гледам по страни и се вълнувам ...,I really look at them from the side and get ex...,excitement,happiness,moderate
4,1900-01-01 00:00:16,1900-01-01 00:00:19,"Ти избра играта, а аз избрах любовта.","You chose the game, and I chose love.",disappointment,sadness,moderate


In [4]:
df, emotions = prepare_data(df, "Translation", "Emotion_core")
print(emotions)
df.head()

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed
['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


,Start Time,End Time,Sentence,Translation,Emotion_fine,Intensity,ekman_emotion,tokenized_text,lemmatized_text
0,1900-01-01 00:00:00,1900-01-01 00:00:04,"Ако приемаш моите извинения, просто ме пригърни.","If you accept my apologies, just hug me.",remorse,moderate,4,"[if, you, accept, my, apologies, ,, just, hug,...","[if, you, accept, my, apology, ,, just, hug, I..."
1,1900-01-01 00:00:06,1900-01-01 00:00:08,"Трябвам това, че си ме пригърни.",I need you to hug me.,longing,moderate,4,"[i, need, you, to, hug, me, .]","[I, need, you, to, hug, I, .]"
2,1900-01-01 00:00:08,1900-01-01 00:00:11,С теб всеки миски е прекрасен. Обичам.,"With you, every moment is wonderful. I love.",affection,moderate,3,"[with, you, ,, every, moment, is, wonderful, ....","[with, you, ,, every, moment, be, wonderful, ...."
3,1900-01-01 00:00:11,1900-01-01 00:00:16,Аз наистина ги гледам по страни и се вълнувам ...,I really look at them from the side and get ex...,excitement,moderate,3,"[i, really, look, at, them, from, the, side, a...","[I, really, look, at, they, from, the, side, a..."
4,1900-01-01 00:00:16,1900-01-01 00:00:19,"Ти избра играта, а аз избрах любовта.","You chose the game, and I chose love.",disappointment,moderate,4,"[you, chose, the, game, ,, and, i, chose, love...","[you, choose, the, game, ,, and, I, choose, lo..."


In [5]:
df.ekman_emotion.value_counts()

ekman_emotion
6    13360
3     6250
4     3745
0     2205
5     2093
2     1932
1      818
Name: count, dtype: int64

In [6]:
df_test = pd.read_csv("../../data/kinga.csv")
df_test, emotions = prepare_data(df_test, "Translation", "Emotion_core")

Detected dataset type: transcript
Starting batch preprocessing...
✓ Text cleaning completed
✓ All NLP processing completed


In [7]:
target_samples = 1000

classes_to_augment = df["ekman_emotion"].value_counts()[df["ekman_emotion"].value_counts() < target_samples].index.tolist()
print(f"Classes to augment: {classes_to_augment}")

Classes to augment: [1]


In [8]:
df_augmented = augment_minority_classes(df, target_samples=target_samples, classes_to_augment=classes_to_augment)
df_augmented = pd.concat([df, df_augmented]).reset_index(drop=True)
print(df_augmented["ekman_emotion"].value_counts())
print(f"Original size: {len(df)}, Augmented size: {len(df_augmented)}")
df = df_augmented

Augmenting emotion 1: 818 → 1000
Augmented Sample 1: it be the bad pizza in denver .
Augmented Sample 2: you a be would fool
Augmented Sample 3: that be a fucking freezer .
ekman_emotion
6    13360
3     6250
4     3745
0     2205
5     2093
2     1932
1     1000
Name: count, dtype: int64
Original size: 30403, Augmented size: 30585


In [9]:
df["lemmatized_text"] = df["lemmatized_text"].apply(lambda tokens: " ".join(tokens))
df_test["lemmatized_text"] = df_test["lemmatized_text"].apply(lambda tokens: " ".join(tokens))

In [10]:
from transformers import pipeline

# sentiment analysis
sentiment = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment", top_k=None)

Device set to use cuda:0


In [11]:
def analyze_sentiment_batch(texts, max_len=500):
    # Trim all texts upfront
    trimmed_texts = [
        t[:max_len] if isinstance(t, str) and len(t) > max_len else t
        for t in texts
    ]

    # Run through the pipeline in batches
    results = sentiment(trimmed_texts, batch_size=256)

    numeric_scores = []
    for result in results:
        best = max(result, key=lambda x: x["score"])
        label = best["label"]
        conf = best["score"]

        if label == "negative":
            numeric_scores.append(-conf)
        elif label == "neutral":
            numeric_scores.append(0.0)
        elif label == "positive":
            numeric_scores.append(conf)
        else:
            numeric_scores.append(0.0)

    return numeric_scores

In [12]:
df["sentiment_score"] = analyze_sentiment_batch(df["lemmatized_text"].tolist())
df_test["sentiment_score"] = analyze_sentiment_batch(df_test["lemmatized_text"].tolist())

In [13]:
# Intensity markers count: occurrences of "very", "so", "extremely" and other similar words
intensity_markers = ["very", "so", "extremely", "highly", "too", "really", "absolutely", "completely", "totally", "utterly"]
df["intensity_marker_count"] = df["lemmatized_text"].apply(lambda x: sum(x.lower().count(marker) for marker in intensity_markers))
df_test["intensity_marker_count"] = df_test["lemmatized_text"].apply(lambda x: sum(x.lower().count(marker) for marker in intensity_markers))

In [14]:
# load emotion lexicon
emotion_lexicon = pd.read_csv("../../data/NRC-Emotion-Lexicon-Wordlevel-v0.92.txt", sep="\t", names=["word", "emotion", "association"])
# pivot the lexicon to have emotions as columns
emotion_lexicon_pivot = emotion_lexicon.pivot(index="word", columns="emotion", values="association").fillna(0)
# create a dictionary for faster lookup
emotion_dict = emotion_lexicon_pivot.to_dict(orient="index")

In [15]:
def return_emotion_counts(lemmas_str):
    lemmas = lemmas_str.lower().split()
    counts = {emotion: 0 for emotion in emotion_lexicon_pivot.columns}

    for lemma in lemmas:
        if lemma in emotion_dict:
            for emotion in emotion_dict[lemma]:
                # increment only emotions marked 1
                if emotion_dict[lemma][emotion] == 1:
                    counts[emotion] += 1

    return pd.Series(counts)

In [16]:
emotion_counts = df["lemmatized_text"].apply(return_emotion_counts)
df = pd.concat([df, emotion_counts], axis=1)

In [17]:
emotion_counts = df_test["lemmatized_text"].apply(return_emotion_counts)
df_test = pd.concat([df_test, emotion_counts], axis=1)

In [18]:
# calculate number of tokens, ratio of unique tokens to total tokens, average word length
df["num_tokens"] = df["lemmatized_text"].apply(lambda x: len(x.split()))
df["unique_token_ratio"] = df["lemmatized_text"].apply(lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0)
df["avg_word_length"] = df["lemmatized_text"].apply(lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0)
df_test["num_tokens"] = df_test["lemmatized_text"].apply(lambda x: len(x.split()))
df_test["unique_token_ratio"] = df_test["lemmatized_text"].apply(lambda x: len(set(x.split())) / len(x.split()) if len(x.split()) > 0 else 0)
df_test["avg_word_length"] = df_test["lemmatized_text"].apply(lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0)

In [19]:
# calculate punctuation count
import string
df["punctuation_count"] = df["lemmatized_text"].apply(lambda x: sum([1 for char in x if char in string.punctuation]))
df_test["punctuation_count"] = df_test["lemmatized_text"].apply(lambda x: sum([1 for char in x if char in string.punctuation]))

In [20]:
# Negation feature: binary if any negation word in window of previous 3 tokens
negation_words = set(["not", "no", "never", "n't", "none", "nobody", "nothing", "neither", "nowhere", "hardly", "scarcely", "barely", "doesn't", "isn't", "wasn't", "shouldn't", "wouldn't", "couldn't", "won't", "can't", "don't"])
def negation_feature(text):
    tokens = text.split()
    return 1 if any(token in negation_words for token in tokens) else 0
df["negation"] = df["lemmatized_text"].apply(negation_feature)
df_test["negation"] = df_test["lemmatized_text"].apply(negation_feature)

In [21]:
from sklearn.preprocessing import StandardScaler
additional_features = df[["sentiment_score", "intensity_marker_count", "num_tokens",
                         "unique_token_ratio", "avg_word_length", "punctuation_count",
                         "negation"] + list(emotion_lexicon_pivot.columns)]

scaler = StandardScaler()
additional_features_scaled = scaler.fit_transform(additional_features)
additional_features_test_scaled = scaler.transform(df_test[["sentiment_score", "intensity_marker_count", "num_tokens",
                                                            "unique_token_ratio", "avg_word_length", "punctuation_count",
                                                            "negation"] + list(emotion_lexicon_pivot.columns)])

In [23]:
additional_features_scaled.shape, additional_features_test_scaled.shape

((30585, 17), (791, 17))

In [24]:
# make sequences additional features to each word vector
def expand_additional_features_to_sequence(additional_features, seq_length):
    expanded_features = []
    for features in additional_features:
        expanded = np.tile(features, (seq_length, 1))  # Repeat features for
        expanded_features.append(expanded)
    return np.array(expanded_features)

In [29]:
additional_features_scaled_seq = expand_additional_features_to_sequence(additional_features_scaled, seq_length=50)
additional_features_test_scaled_seq = expand_additional_features_to_sequence(additional_features_test_scaled, seq_length=50)
additional_features_scaled_seq.shape, additional_features_test_scaled_seq.shape

((30585, 50, 17), (791, 50, 17))

In [30]:
word2vec_model = api.load("word2vec-google-news-300")

In [31]:
X_train, y_train = prepare_rnn_data(df, "lemmatized_text", word2vec_model, "ekman_emotion")
X_test, y_test = prepare_rnn_data(df_test, "lemmatized_text", word2vec_model, "ekman_emotion")
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Preparing RNN data from 30585 samples...
Embedding dimension: 300
Max sequence length: 50


Converting to embeddings: 100%|██████████| 30585/30585 [00:01<00:00, 20936.18it/s]


✓ Final shape: (30585, 50, 300)
✓ OOV rate: 23.48% (57702/245754)
✓ Data type: float64
Preparing RNN data from 791 samples...
Embedding dimension: 300
Max sequence length: 50


Converting to embeddings: 100%|██████████| 791/791 [00:00<00:00, 21962.76it/s]

✓ Final shape: (791, 50, 300)
✓ OOV rate: 27.23% (1711/6283)
✓ Data type: float64
Train shape: (30585, 50, 300), Test shape: (791, 50, 300)


In [32]:
# combine word vectors with additional features
X_train = np.concatenate((X_train, additional_features_scaled_seq), axis=2)
X_test = np.concatenate((X_test, additional_features_test_scaled_seq), axis=2)
print(f"Combined Train shape: {X_train.shape}, Combined Test shape: {X_test.shape}")

Combined Train shape: (30585, 50, 317), Combined Test shape: (791, 50, 317)


In [33]:
class EmotionDataset(Dataset):
    """PyTorch Dataset for emotion classification"""

    def __init__(self, sequences, labels):
        self.sequences = torch.FloatTensor(sequences)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [34]:
class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.3):
            super().__init__()

            # Input normalization
            self.input_norm = nn.LayerNorm(input_size)

            # BiLSTM with proper initialization
            self.lstm = nn.LSTM(
                input_size,
                hidden_size,
                num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0,
                bidirectional=True
            )

            # Attention mechanism (helps with long sequences)
            self.attention = nn.MultiheadAttention(
                embed_dim=hidden_size * 2,
                num_heads=8,
                dropout=dropout,
                batch_first=True
            )

            # Classification layers
            self.dropout = nn.Dropout(dropout)
            self.batch_norm = nn.BatchNorm1d(hidden_size * 2)

            # Simpler classifier
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size * 2, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_size, num_classes)
            )

            # Initialize weights properly
            self._init_weights()

    def _init_weights(self):
            """Proper weight initialization"""
            for name, param in self.named_parameters():
                if 'weight' in name:
                    if 'lstm' in name:
                        nn.init.xavier_uniform_(param)
                    elif 'linear' in name or 'fc' in name:
                        nn.init.xavier_uniform_(param)
                elif 'bias' in name:
                    nn.init.constant_(param, 0)

    def forward(self, x):
            # Input normalization
            x = self.input_norm(x)

            # BiLSTM
            lstm_out, (hidden, cell) = self.lstm(x)
            lstm_out = self.dropout(lstm_out)

            # Attention mechanism (instead of just taking last output)
            attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)

            # Global max pooling across sequence dimension
            pooled = torch.max(attn_out, dim=1)[0]  # Take max across sequence

            # Batch normalization
            pooled = self.batch_norm(pooled)

            # Classification
            output = self.classifier(pooled)

            return output

In [35]:
# train / validation split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train, shuffle=True)

X_train = np.array(X_train)
y_train = np.array(y_train)
X_val = np.array(X_val)
y_val = np.array(y_val)

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [37]:
num_classes = len(emotions)
epochs = 250
patience = 10
lr_patience = 10
model_save_path = "../../models/BiLSTM/best_model.pth"
num_classes

7

In [41]:
def create_better_training_setup(input_size=X_train.shape[2], num_classes=num_classes):
    """
    Create better training configuration
    """

    training_config = {
        # Model architecture
        'input_size': input_size,
        'hidden_size': 128,  # Smaller hidden size
        'num_layers': 1,     # Fewer layers
        'dropout': 0.2,      # Lower dropout

        # Training parameters
        'learning_rate': 0.001,   # Higher learning rate
        'weight_decay': 1e-5,     # Lower weight decay
        'batch_size': 64,         # Smaller batch size
        'patience': 15,           # More patience
        'lr_patience': 5,         # Reduce LR sooner

        # Class weights (more aggressive for minorities)
        'custom_class_weights': compute_class_weight(
            class_weight='balanced',
            classes=np.arange(num_classes),
            y=y_train
        ).tolist()
    }

    return training_config

In [48]:
def train_bilstm(X_train, X_val, X_test, y_train, y_val, y_test, emotions, device, save_path=model_save_path):
    """
    Train BiLSTM with all fixes applied
    """

    print("TRAINING FIXED BILSTM MODEL")
    print("="*35)

    # Get fixed training configuration
    config = create_better_training_setup(input_size=X_train.shape[2], num_classes=len(emotions))

    # Create datasets with better batch size
    train_dataset = EmotionDataset(X_train, y_train)
    val_dataset = EmotionDataset(X_val, y_val)
    test_dataset = EmotionDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

    # Create fixed model
    model = BiLSTM(
        input_size=config['input_size'],
        hidden_size=config['hidden_size'],
        num_layers=config['num_layers'],
        num_classes=len(emotions),
        dropout=config['dropout']
    ).to(device)

    print(f"Model architecture:")
    print(f"  Hidden size: {config['hidden_size']}")
    print(f"  Num layers: {config['num_layers']}")
    print(f"  Dropout: {config['dropout']}")
    print(f"  Batch size: {config['batch_size']}")

    # Better class weights
    class_weights = torch.FloatTensor([
        config['custom_class_weights'][i] for i in range(len(emotions))
    ]).to(device)

    print(f"Class weights: {class_weights.cpu().numpy()}")

    # Better optimizer and loss
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)  # Add label smoothing
    optimizer = optim.AdamW(  # Use AdamW instead of Adam
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    # Better scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )

    print(f"Learning rate: {config['learning_rate']}")
    print(f"Weight decay: {config['weight_decay']}")
    print(f"Using label smoothing: 0.1")

    # Training loop with better monitoring
    best_val_acc = 0
    patience_counter = 0
    overfitting_threshold = 15  # %
    overfitting_patience = 5
    overfitting_counter = 0

    for epoch in range(500):  # Fewer max epochs, focus on quality
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        if overfitting_counter > overfitting_patience:
            print("⚠️  Stopping training due to overfitting.")
            break

        for sequences, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
            sequences, labels = sequences.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)

            # Check for NaN
            if torch.isnan(loss):
                print("⚠️  NaN loss detected! Stopping training.")
                break

            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_labels = []

        with torch.no_grad():
            for sequences, labels in val_loader:
                sequences, labels = sequences.to(device), labels.to(device)
                outputs = model(sequences)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_val_preds.extend(predicted.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        # Calculate metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = 100 * train_correct / train_total
        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = 100 * val_correct / val_total
        current_lr = optimizer.param_groups[0]['lr']

        is_overfitting = epoch_train_acc - epoch_val_acc > overfitting_threshold
        if is_overfitting:
            overfitting_counter += 1
            print(f"⚠️  Overfitting detected! ({overfitting_counter}/{overfitting_patience})")
        else:
            overfitting_counter = 0

        print(f"Epoch {epoch+1:2d}: Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:5.2f}%, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:5.2f}%, LR: {current_lr:.2e}")

        # Check prediction diversity
        unique_preds = len(set(all_val_preds))
        print(f"           Predicting {unique_preds}/7 different emotions")

        # Early stopping with improvement
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), save_path)
            print(f"Saved best model with Val Acc: {best_val_acc:.2f}% on epoch {epoch+1}")
        else:
            patience_counter += 1

        if patience_counter >= config['patience']:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Load best model and evaluate
    model.load_state_dict(torch.load(save_path))

    # Final test evaluation
    test_dataset = EmotionDataset(X_test, y_test)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)

    model.eval()
    all_test_preds = []
    all_test_labels = []

    with torch.no_grad():
        for sequences, labels in test_loader:
            sequences, labels = sequences.to(device), labels.to(device)
            outputs = model(sequences)
            _, predicted = torch.max(outputs, 1)

            all_test_preds.extend(predicted.cpu().numpy())
            all_test_labels.extend(labels.cpu().numpy())

    test_accuracy = sum(p == l for p, l in zip(all_test_preds, all_test_labels)) / len(all_test_preds)

    print(f"\nFINAL FIXED MODEL RESULTS:")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(all_test_labels, all_test_preds, target_names=emotions, zero_division=0, digits=3))

    return model, all_test_preds

In [49]:
model, preds = train_bilstm(X_train, X_val, X_test, y_train, y_val, y_test, emotions, device)

TRAINING FIXED BILSTM MODEL
Model architecture:
  Hidden size: 128
  Num layers: 1
  Dropout: 0.2
  Batch size: 64
Class weights: [1.9819988 4.3692064 2.2612338 0.699073  1.1668503 2.0872004 0.3270364]
Learning rate: 0.001
Weight decay: 1e-05
Using label smoothing: 0.1


Epoch 1: 100%|██████████| 431/431 [00:01<00:00, 231.67it/s]


Epoch  1: Train Loss: 1.9023, Train Acc: 38.75%, Val Loss: 1.8375, Val Acc: 45.80%, LR: 6.87e-04
           Predicting 7/7 different emotions
Saved best model with Val Acc: 45.80% on epoch 1


Epoch 2: 100%|██████████| 431/431 [00:01<00:00, 266.17it/s]


Epoch  2: Train Loss: 1.7991, Train Acc: 45.50%, Val Loss: 1.7245, Val Acc: 48.02%, LR: 7.10e-04
           Predicting 7/7 different emotions
Saved best model with Val Acc: 48.02% on epoch 2


...

⚠️  Overfitting detected! (4/5)
Epoch 19: Train Loss: 0.9322, Train Acc: 84.95%, Val Loss: 2.1460, Val Acc: 63.16%, LR: 3.44e-04
           Predicting 7/7 different emotions
Saved best model with Val Acc: 63.16% on epoch 19


Epoch 20: 100%|██████████| 431/431 [00:01<00:00, 262.73it/s]


⚠️  Overfitting detected! (5/5)
Epoch 20: Train Loss: 0.8951, Train Acc: 87.05%, Val Loss: 2.1260, Val Acc: 62.77%, LR: 2.26e-04
           Predicting 7/7 different emotions


Epoch 21: 100%|██████████| 431/431 [00:01<00:00, 245.63it/s]


⚠️  Overfitting detected! (6/5)
Epoch 21: Train Loss: 0.8710, Train Acc: 88.91%, Val Loss: 2.1840, Val Acc: 62.05%, LR: 1.26e-04
           Predicting 7/7 different emotions
⚠️  Stopping training due to overfitting.

FINAL FIXED MODEL RESULTS:
Test Accuracy: 0.6422

Classification Report:
              precision    recall  f1-score   support

       anger      0.375     0.429     0.400        35
     disgust      0.333     0.444     0.381         9
        fear      0.558     0.500     0.527        58
         joy      0.708     0.523     0.602       153
     sadness      0.403     0.482     0.439        56
    surprise      0.424     0.418     0.421        67
     neutral      0.737     0.787     0.761       413

    accuracy                          0.642       791
   macro avg      0.505     0.512     0.504       791
weighted avg      0.647     0.642     0.641       791

